# PC-FNO3D NATIONAL v4 — lecture UNIQUE du test 2022 (Colab, A100)

Règle `configs/national/regle_v4_national.json` (sha256 `60671e843ee6…`), vérifiée par le carnet. **Uniquement avec l'accord écrit d'Ines.** Vérité d'entraînement v3 (`cache_v3_partie_*`).

L'entraînement est choisi sur le score de validation des six (C' et D, 3 graines chacun) ; le test n'est lu que pour lui, une fois, et pour le modèle livré sur la même vérité.

## 0. GPU (A100 obligatoire)

In [ ]:
# Le national (453 M parametres, grille 16 x 828 x 388) ne tient pas sur une L4 de 22 Go, meme avec le
# recalcul des activations (3 echecs le 11/11, dont un avec le carnet corrige : 20,8 Go + 2,45 Go demandes).
# Si Colab n'attribue pas un GPU d'au moins 30 Go (A100), la session est liberee AVANT le montage du Drive.
import subprocess as _sp
_g = _sp.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
             capture_output=True, text=True)
_nom, _mo = ([s.strip() for s in _g.stdout.strip().splitlines()[0].split(",")] if _g.returncode == 0
             and _g.stdout.strip() else ("aucun GPU", "0"))
print("GPU attribue :", _nom, f"{float(_mo) / 1024:.0f} Go")
if float(_mo) < 30 * 1024:
    from google.colab import runtime
    print("GPU trop petit pour le national : session liberee, rien n'a ete lance. Relancer plus tard (A100).")
    runtime.unassign()
    raise SystemExit("GPU de moins de 30 Go")

## 1. Drive, code et données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

# code du depot public au commit fige (remplace l'archive du Drive, plus ancienne)
COMMIT = "77a058b0649f77648837559ba3f9b807cbe5a41f"
os.chdir("/content")    # le noyau etait dans /content/PFE : le supprimer laisserait pip dans un dossier efface
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
os.chdir("/content/PFE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install -e /content/PFE a echoue (sortie ci-dessus)")
import importlib
importlib.invalidate_caches()
import hashlib
for f, h in (("regle_v4_national.json", "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8"),):
    assert hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest() == h, f
print("code", COMMIT, "| regle v4 verifiee")

## 2. Cache v3

In [ ]:
LOC = Path("/content/national")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (NAT / f"cache_v3_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_v3_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(NAT / "jeu_v3.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())
lancer("controle", "python -u -m pcfno3d -c national_1km_colab check")

## 3. Mémoire du GPU

In [ ]:
# Essai de memoire dans un PROCESSUS SEPARE : le noyau du carnet ne touche jamais au GPU (le 11/11, l'essai fait
# dans le noyau y gardait 3,3 Go et l'entrainement sur L4 22 Go a manque de memoire). Allocateur a segments
# extensibles (moins de fragmentation), herite par les commandes lancees ensuite. Aucun effet sur les calculs.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
ESSAI = r"""
import json, sys, torch
from pcfno3d.config import Config
from pcfno3d.geometry import Geometry
from pcfno3d.model import build_model
recalcul = sys.argv[1] == "1"
cfg = Config.load("configs/national_1km_colab.yaml")
cfg.model.recalcul_activations = recalcul
geom = Geometry(cfg, device="cuda")
m = build_model(cfg, geom.c_in).cuda().train()
x = torch.randn(cfg.train.batch_size, geom.c_in, *geom.shape, device="cuda")
torch.cuda.reset_peak_memory_stats()
try:
    with torch.autocast("cuda", dtype=torch.bfloat16):
        m(x).float().pow(2).mean().backward()
    pic = torch.cuda.max_memory_allocated() / 1e9
except torch.cuda.OutOfMemoryError:
    pic = None
p = torch.cuda.get_device_properties(0)
print("ESSAI", json.dumps({"gpu": p.name, "total": p.total_memory / 1e9, "pic": pic}))
"""
Path("/content/essai_memoire.py").write_text(ESSAI)
def essai(recalcul: bool) -> dict:
    r = subprocess.run([sys.executable, "/content/essai_memoire.py", "1" if recalcul else "0"], cwd="/content/PFE",
                       capture_output=True, text=True)
    ligne = next((l for l in r.stdout.splitlines() if l.startswith("ESSAI ")), None)
    if ligne is None:
        print(r.stdout[-2000:], r.stderr[-2000:])
        return {"gpu": "?", "total": float("nan"), "pic": None}
    return json.loads(ligne[6:])
e = essai(False)
total = e["total"]
RECALCUL = e["pic"] is None or e["pic"] > 0.85 * total
if RECALCUL:
    e = essai(True)
print(e["gpu"], f"{total:.0f} Go | pic de l'essai {e['pic'] if e['pic'] is None else round(e['pic'], 1)} Go "
      f"| recalcul des activations : {RECALCUL}")
if total < 30:
    print("ATTENTION : GPU de moins de 30 Go (L4 ?). Le national a manque de memoire sur L4 le 11/11 avec l'ancien "
          "carnet ; cet essai ne compte ni l'optimiseur ni l'EMA. Preferer une A100.")
OPTS = f"--set model.recalcul_activations={str(RECALCUL).lower()}"

## 4. Choix de l'entraînement, test, DSO, mesures réelles, décision

In [ ]:
import shlex
def choisir_poids(src):
    """Le Drive garde parfois deux fichiers du meme nom (grand_tunis_v4_dso : deux best.pt de 1,69 Go,
    vus le 31/10) ; Colab peut n'en montrer qu'un, au hasard. On retient celui dont l'epoque est celle
    du test evalue (eval_pcfno3d/best_wcont, sinon best), meme s'il n'y en a qu'un : jamais un poids
    pris au hasard."""
    cands = sorted(src.glob("best*.pt"))
    if not cands:
        return src / "best.pt"
    attendu = next((json.loads(f.read_text()).get("epoque") for d in ("best_wcont", "best")
                    if (f := src / "eval_pcfno3d" / d / "test_summary.json").exists()), None)
    import torch
    epoques = {}
    for c in cands:
        ck = torch.load(str(c), map_location="cpu", mmap=True, weights_only=False)
        epoques[c] = ck.get("epoch", -2) + 1
        del ck
    print("poids candidats :", {c.name: e for c, e in epoques.items()}, "| epoque evaluee :", attendu)
    ok = [c for c in cands if epoques[c] == attendu] if attendu is not None else cands
    if not ok:
        raise RuntimeError(f"aucun poids de {src} n'a l'epoque evaluee ({attendu}) : verifie sur le Drive")
    return max(ok, key=lambda c: c.stat().st_mtime)          # a epoque egale : le plus recent

import shlex
from google.colab import runtime
import traceback
regle = json.loads((CODE / "configs" / "national" / "regle_v4_national.json").read_text(encoding="utf-8"))
scores = {}
for cand in ("Cp", "D"):
    for g in regle["candidats"]["commun"]["graines"]:
        h = NAT / "runs" / f"national_1km_v4_{cand}_s{g}" / "history.json"
        if not h.exists():
            raise FileNotFoundError(f"{cand} graine {g} pas entraine ({h})")
        scores[f"{cand}_s{g}"] = min(float(e["val_score"]) for e in json.loads(h.read_text()) if "val_score" in e)
print("scores de selection (validation) :", scores)
GRAINE = min(scores, key=scores.get)          # ex. "Cp_s43" : l'entrainement choisi
RUN = NAT / "runs" / f"national_1km_v4_{GRAINE}"
import statistics
par_cand = {c: [v for k, v in scores.items() if k.startswith(c + "_")] for c in ("Cp", "D")}
choix = {"regle_sha256": "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8", "scores_validation": scores, "entrainement_choisi": GRAINE,
         "par_candidat": {c: {"moyenne": statistics.mean(v), "ecart_type": statistics.pstdev(v)}
                          for c, v in par_cand.items()}}
(NAT / "runs" / "choix_v4.json").write_text(json.dumps(choix, indent=2))
import yaml
c = yaml.safe_load(open(CODE / "configs" / "national_1km_colab.yaml"))
c["name"] = "national_v4_eval"
c["paths"]["run_dir"], c["paths"]["local_dir"] = str(RUN), f"/content/run_local_v4_s{GRAINE}"
yaml.safe_dump(c, open(CODE / "configs" / "national_v4_eval.yaml", "w"), sort_keys=False)
O = OPTS
try:
    if not (RUN / "eval_pcfno3d" / "best_wcont" / "test_summary.json").exists():   # test lu une fois
        lancer("evaluer_v4", f"python -u -m pcfno3d -c national_v4_eval {O} evaluate")
    CAL = RUN / "fiabilite_pcfno3d" / "best_wcont" / "calibration_validation.json"
    D6 = RUN / "eval_pcfno3d" / "best_wcont" / "dso6_modele_livre.json"
    if not CAL.exists():
        lancer("calibrer_dso6_v4", f"python -u -m pcfno3d -c national_v4_eval {O} calibrate-reliability --out {CAL}")
    if not D6.exists():
        try:
            lancer("dso6_v4", f"python -u -m pcfno3d -c national_v4_eval {O} dso6 --calibration {CAL} --out {D6}")
        except RuntimeError as exc:
            if "(code 3)" not in str(exc):
                raise
            print(exc)
    if not (RUN / "validation_reelle" / "test" / "bilan.json").exists():
        lancer("validation_reelle_v4", f"python -u scripts/valider_grand_tunis_reel.py -c national_v4_eval "
                                       f"--obs {G / 'obs'} --jeu test")
    for cmd, nom in ((f"python -u -m pcfno3d -c national_v4_eval {O} verdict-national", "verdict_national_v4"),
                     ("python -u scripts/dso_hauteur_sol.py -c national_v4_eval regle", "dso_sol_regle_v4"),
                     ("python -u scripts/dso_hauteur_sol.py -c national_v4_eval mesure", "dso_sol_v4"),
                     ("python -u scripts/dso_hauteur_sol.py -c national_v4_eval mesure --keep 0.25", "dso_sol_dso5_v4")):
        try:
            lancer(nom, cmd)
        except RuntimeError as exc:
            if "(code 3)" not in str(exc):
                raise
            print(exc)
    # modele livre (v2, poids inchanges) sur la MEME verite v3, dossier neuf (regle_v4_national.json)
    REF = NAT / "runs" / "national_1km_v2_sur_verite_v3"
    REF.mkdir(parents=True, exist_ok=True)
    V2 = NAT / "runs" / "national_1km_v2"
    for src, dst in ((choisir_poids(V2), REF / "best.pt"), (V2 / "norm_stats.json", REF / "norm_stats.json")):
        if not dst.exists():
            shutil.copy2(src, dst)
    c2 = yaml.safe_load(open(CODE / "configs" / "national_1km_colab.yaml"))
    c2["name"] = "national_v2_sur_v3_eval"
    c2["paths"]["run_dir"], c2["paths"]["local_dir"] = str(REF), "/content/run_local_v2_sur_v3"
    yaml.safe_dump(c2, open(CODE / "configs" / "national_v2_sur_v3_eval.yaml", "w"), sort_keys=False)
    if not (REF / "eval_pcfno3d" / "best_wcont" / "test_summary.json").exists():   # test lu une fois
        lancer("evaluer_v2_sur_v3", f"python -u -m pcfno3d -c national_v2_sur_v3_eval {O} evaluate")
    if not (REF / "validation_reelle" / "test" / "bilan.json").exists():
        lancer("validation_reelle_v2_sur_v3", f"python -u scripts/valider_grand_tunis_reel.py -c national_v2_sur_v3_eval "
                                              f"--obs {G / 'obs'} --jeu test")
    for cmd, nom in ((f"python -u -m pcfno3d -c national_v2_sur_v3_eval {O} verdict-national", "verdict_national_v2_sur_v3"),
                     ("python -u scripts/dso_hauteur_sol.py -c national_v2_sur_v3_eval regle", "dso_sol_regle_v2_sur_v3"),
                     ("python -u scripts/dso_hauteur_sol.py -c national_v2_sur_v3_eval mesure", "dso_sol_v2_sur_v3"),
                     ("python -u scripts/dso_hauteur_sol.py -c national_v2_sur_v3_eval mesure --keep 0.25",
                      "dso_sol_dso5_v2_sur_v3")):
        try:
            lancer(nom, cmd)
        except RuntimeError as exc:
            if "(code 3)" not in str(exc):
                raise
            print(exc)
    try:
        lancer("decision_v4_national", f"python -u scripts/decision_v4_national.py --v4 {RUN} --livre {REF} "
                                       f"--sortie {NAT / 'runs'}")
    except RuntimeError as exc:
        if "(code 3)" not in str(exc):
            raise
        print(exc)
    for f in sorted(RUN.rglob("bilan.json")) + sorted(RUN.rglob("verdict_national.json"))             + [NAT / "runs" / "decision_v4_national.json"]:
        print(f); print(f.read_text()[:3000])
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()